# 02_ingestion — Chargement des six sources vers Delta
Lecture de chaque source depuis le volume `landing`, puis écriture dans une table Delta managée `alpenia_dev.bronze.brz_<source>`.


In [ ]:
BASE = "/Volumes/alpenia_dev/bronze/landing"
TARGET = "alpenia_dev.bronze"

def check(df, name):
    """Affiche le schéma et le nombre de lignes juste après la lecture."""
    print(f"===== {name} : {df.count():,} lignes")
    df.printSchema()

## 1. Lecture des sources
Chaque format a ses options : un lecteur mal paramétré ne lève pas toujours d'erreur, il produit des données fausses.

### contracts — CSV, séparateur `;`
Sans `sep=";"`, Spark lirait une seule colonne contenant toute la ligne.

In [ ]:
df_contracts = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .load(f"{BASE}/contracts/"))
check(df_contracts, "contracts")

### clients et pricing — CSV standard (séparateur `,`)

In [ ]:
df_clients = (spark.read
    .format("csv")
    .option("header", "true")
    .load(f"{BASE}/clients/"))
check(df_clients, "clients")

df_pricing = (spark.read
    .format("csv")
    .option("header", "true")
    .load(f"{BASE}/pricing/"))
check(df_pricing, "pricing")

### claims — Parquet, 3 fichiers mensuels
Parquet embarque son schéma (types inclus) : pas d'option nécessaire. Lire le **dossier** charge les trois fichiers d'un coup.

In [ ]:
df_claims = (spark.read
    .format("parquet")
    .load(f"{BASE}/claims/"))
check(df_claims, "claims")

### web_events — JSON Lines (un objet par ligne), structure imbriquée
C'est le mode par défaut du lecteur JSON (`multiLine=false`). La colonne `quote` arrive en `STRUCT`.

In [ ]:
df_web_events = (spark.read
    .format("json")
    .load(f"{BASE}/web_events/"))
check(df_web_events, "web_events")

### expertises — JSON multi-lignes (un tableau par fichier)
Sans `multiLine=true`, Spark tente de lire chaque ligne physique comme un objet et renvoie une colonne `_corrupt_record`.

In [ ]:
df_expertises = (spark.read
    .format("json")
    .option("multiLine", "true")
    .load(f"{BASE}/expertises/"))
check(df_expertises, "expertises")
assert "_corrupt_record" not in df_expertises.columns, "Lecture JSON incorrecte : vérifier multiLine" 

## 2. Écriture en tables Delta managées

Mode retenu : `overwrite`. Chaque exécution remplace intégralement le contenu de la table par le contenu actuel des fichiers. Comme chaque dossier contient l'export complet de sa source, relancer le notebook produit exactement le même résultat : c'est ce qui rend le chargement idempotent. Avec `append`, chaque relance ajouterait une copie complète des données (doublons).

`overwriteSchema=true` : `03_transformations` change le type des colonnes (STRING → DATE, etc.). Sans cette option, relancer `02_ingestion` après `03` échouerait, car Delta refuse par défaut de remplacer une table par des données d'un autre schéma. Ici, on veut explicitement revenir au schéma brut.

In [ ]:
sources = {
    "brz_contracts":  df_contracts,
    "brz_clients":    df_clients,
    "brz_pricing":    df_pricing,
    "brz_claims":     df_claims,
    "brz_web_events": df_web_events,
    "brz_expertises": df_expertises,
}

for table, df in sources.items():
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{TARGET}.{table}"))
    print(f"{TARGET}.{table} écrite")

## 3. Contrôle d'idempotence
Nombre de lignes attendu (énoncé, partie 2) vs nombre de lignes présent dans chaque table. Relancer le notebook entier : les chiffres doivent rester strictement identiques.

In [ ]:
from datetime import datetime

EXPECTED_ROWS = {
    "brz_contracts":  4000,
    "brz_clients":    3000,
    "brz_pricing":    6,
    "brz_claims":     24000,
    "brz_web_events": 90000,
    "brz_expertises": 450,
}

control = []
for table, expected in EXPECTED_ROWS.items():
    actual = spark.table(f"{TARGET}.{table}").count()
    version = spark.sql(f"DESCRIBE HISTORY {TARGET}.{table} LIMIT 1").first()["version"]
    control.append((table, expected, actual, "OK" if actual == expected else "ÉCART", version))

df_control = spark.createDataFrame(
    control, "table STRING, expected_rows LONG, actual_rows LONG, status STRING, delta_version LONG")
print("Contrôle exécuté le", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
display(df_control)

assert all(r[3] == "OK" for r in control), "Au moins une table n'a pas le nombre de lignes attendu"
